In [ ]:
# --- Ensure working directory is project root (contains 'parameter/') ---
import os
if not os.path.isdir('parameter') and os.path.isdir('../parameter'):
    os.chdir('..')


In [ ]:
# --- Pick per-case data via CASE_ID env var ---
import os
CASE_ID = os.environ.get('CASE_ID')
if CASE_ID is None:
    raise RuntimeError("CASE_ID env var must be set (e.g. 'case0_N5').")
print(f'Running case: {CASE_ID}')


In [ ]:
import numpy as np
from numpy.linalg import norm
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import cvxpy as cp
import daqp
from matplotlib.patches import Polygon,Circle
import pickle
import time



 # Based on 1124_fvo_cbf_no_divide_zero from Nov 24, now demonstrating it works across various initial conditions
 # Currently generating many cases and plugging them in one by one

 # This version adds position/speed std dev for the initial conditions and for the final step
 


In [ ]:

# ==============================================================================
# 1. System dynamics definition (multi-agent)
# ==============================================================================

def multi_agent_const_speed_5d_dynamics(t, y, u_all_flat, n_agents, state_dim, V_const):
    """
    Dynamics of N_AGENTS 5D constant-speed models.
    State: [px, py, vx, vy, theta], control: [omega]

    Args:
        t (float): time
        y (np.ndarray): state of all agents (n_agents * 5,1)
        u_all_flat (np.ndarray): control inputs for all agents (n_agents * 1,)
        n_agents (int): number of agents
        state_dim (int): state dimension (must be 5)
        V_const (float): fixed speed shared by all agents

    Returns:
        np.ndarray: time derivative of the state
    """
    # 1. Reshape data
    # y is a 1D vector; current_states is shape (5, n_agents) 
    current_states = y.reshape((state_dim, n_agents), order='F')
    control_inputs = u_all_flat.reshape((1, n_agents), order='F')

    # 2. Extract state/control variables; also needs to rewrite position and heading initialization
    vx = current_states[2, :]     # (fix) extract vx, vy since they are needed for computation
    vy = current_states[3, :]
    theta = current_states[4, :]
    omega = control_inputs[0, :]

    # 3. Compute state derivative (d_state/dt) (updated physics)
    d_state = np.zeros_like(current_states)
    
    # Position derivative is velocity
    d_state[0, :] = vx
    d_state[1, :] = vy
    
    # Velocity derivative is acceleration (apply chain rule)
    # d(vx)/dt = d/dt(V*cos(theta)) = -V*sin(theta)*(d(theta)/dt) = -V*sin(theta)*omega
    # d(vy)/dt = d/dt(V*sin(theta)) =  V*cos(theta)*(d(theta)/dt) =  V*cos(theta)*omega
    d_state[2, :] = -V_const * np.sin(theta) * omega
    d_state[3, :] =  V_const * np.cos(theta) * omega
    
    # Angular rate equals angular velocity
    d_state[4, :] = omega
    

    # 4. Return result (flattened to 1D vector for solve_ivp)
    return d_state.flatten('F')


In [ ]:

def relative_state(STATE_DIM, N_AGENTS, current_states_matrix):
    """
    Vectorized version of the function that computes relative states between all agent pairs.
    Removes the double for loop and uses NumPy broadcasting for better performance.
    """
    
    # --- 1. Split state vectors needed for computation ---
    # p: positions (2, N_AGENTS), v: velocities (2, N_AGENTS), seta: angles (N_AGENTS,)
    p = current_states_matrix[0:2, :]
    v = current_states_matrix[2:4, :]
    seta = current_states_matrix[4, :]

    # --- 2. Compute relative vectors for all pairs via broadcasting ---
    # By reshaping p to (2, N, 1) and p_T to (2, 1, N) and subtracting,
    # yields a (2, N, N) tensor whose [:, i, j] element is (p_i - p_j)
    relative_p_tensor = p[:, :, np.newaxis] - p[:, np.newaxis, :]
    relative_v_tensor = v[:, :, np.newaxis] - v[:, np.newaxis, :]

    # --- 3. Compute norms for all pairs ---
    # axis=0 means compute the norm over the 2D coordinates (x, y)
    norm_relative_p = np.linalg.norm(relative_p_tensor, axis=0)
    norm_relative_v = np.linalg.norm(relative_v_tensor, axis=0)

    # --- 4. Compute dot products for all pairs ---
    # Elementwise multiply the two tensors, then sum over the coordinate axis (axis=0) to get the dot product
    dot_relative_pv = np.sum(relative_p_tensor * relative_v_tensor, axis=0)

    # --- 5. Compute relative angles (seta) for all pairs ---
    # Reshape seta to (N, 1) and (1, N) for broadcasting
    relative_seta = np.sin(seta[np.newaxis, :] - seta[:, np.newaxis])

    # --- 6. Compute dot_relative_p_seta for all pairs ---
    # Compute vector_seta for all i at once
    vector_seta_all = np.vstack([-np.sin(seta), np.cos(seta)]) # shape: (2, N_AGENTS)

    # Reshape vector_seta_all to (2, N, 1) and multiply with relative_p_tensor
    # Sum along axis=0 for the dot product
    # Original expression: dot(vector_seta_i, -(p_i - p_j)) = dot(vector_seta_i, p_j - p_i)
    dot_relative_p_seta = np.sum(vector_seta_all[:, :, np.newaxis] * (-relative_p_tensor), axis=0)

    return norm_relative_p, norm_relative_v, dot_relative_pv, relative_seta, dot_relative_p_seta

In [ ]:

def ACS_flocking(u_limit, beta, lamda, k1, k2, desired_distance, N_AGENTS, V_CONST, pij_norm, vij_norm, pv_norm, seta_ij, p_dot_seta):
    """
    Vectorized version of the ACS_flocking algorithm.
    Removes the for loop and computes control inputs for all agents simultaneously using NumPy array operations.

    Key changes:
    - Instead of np.delete to exclude self-interactions (i=j),
      we instead zero out the matrix diagonal so those entries are excluded from sum().
    - Set the diagonal to 1 temporarily to avoid division-by-zero.
    - Replace if-statements with np.clip.
    """

    # --- 1. Alignment Term ---
    
    # Compute weights for all agent pairs at once
    # Flocking original
    #align_weight_matrix = (1 + (pij_norm)**2)**(-beta) 

    align_weight_matrix = (1 + (pij_norm)**2)**(-beta)
    
    # For each agent i, compute the weighted sum over other agents j (axis=1 sums along rows)
    u_align_all = (lamda / N_AGENTS) * np.sum(align_weight_matrix * seta_ij, axis=1)

    # --- 2. Inter-force Term ---
    # The diagonal of pij_norm is zero, so to avoid division-by-zero errors
    # Create a copy with the diagonal temporarily replaced by 1.0.
    # The diagonal is zeroed in the final computation so it does not affect the sum -- safe.
    pij_norm_safe = pij_norm.copy()
    np.fill_diagonal(pij_norm_safe, 1.0)

    # Compute each part of the interaction force at the matrix level in one pass
    term_A_matrix = (k1 / (2 * pij_norm_safe**2)) * pv_norm
    term_B_matrix = (k2 * (pij_norm_safe - desired_distance) / (2 * pij_norm_safe)) * p_dot_seta
    
    # Combine the two terms
    combined_matrix = term_A_matrix + term_B_matrix
    
    # Zero the diagonal to exclude self-interactions (i=j)
    np.fill_diagonal(combined_matrix, 0)
    
    # Sum the interaction forces from all other agents j for each agent i
    u_inter_all = (1 / (N_AGENTS * V_CONST)) * np.sum(combined_matrix, axis=1)

    # --- 3. Compute final control input and apply saturation ---
    # Add the alignment term and the interaction force term
    flocking_control = u_align_all + u_inter_all
    
    # Saturation: replace if-statement with np.clip to handle in one shot
    flocking_control = np.clip(flocking_control, -u_limit, u_limit)
    
    # Truncation
    flocking_control = np.trunc(flocking_control * 1000) / 1000

    # Reshape the final output to (N_AGENTS, 1)
    return flocking_control.reshape(-1, 1)

In [ ]:
def minimum_inter_distance_function(pij_norm, N_AGENTS):
    # Copy to avoid mutating the original pij_norm matrix.
    pij_with_inf_diag = pij_norm.copy()
    
    # Fill the diagonal of the copied matrix with np.inf.
    np.fill_diagonal(pij_with_inf_diag, np.inf)
    
    # Find the minimum over the entire matrix.
    return np.min(pij_with_inf_diag)

In [ ]:
def maximum_inter_distance_function(pij_norm, N_AGENTS):
    # Copy to avoid mutating the original pij_norm matrix.
    pij_with_inf_diag = pij_norm.copy()
    
    # Fill the diagonal of the copied matrix with np.inf.
    np.fill_diagonal(pij_with_inf_diag, -1*np.inf)
    
    # Find the minimum over the entire matrix.
    return np.max(pij_with_inf_diag)

In [ ]:
def vo_cbf(pij_norm, vij_norm, pv_norm, N_AGENTS, V_CONST, critical_distance, current_states_matrix, margin):
    """
    Optimized vectorized version of the Control Barrier Function (CBF) computation.
    Uses conditional masking (Boolean Indexing) to perform only the necessary computations.
    """
    np.seterr(divide='warn', invalid='warn')

    
    # --- 1. Initial setup and index computation for all agent pairs ---
    matrix_size = (N_AGENTS * (N_AGENTS - 1)) // 2
    rows, cols = np.triu_indices(N_AGENTS, k=1)

    seta = current_states_matrix[4, :]
    p_vector = current_states_matrix[0:2, :]
    v_vector = current_states_matrix[2:4, :]

    # --- 2. Pre-compute relative state vectors and values for all pairs ---
    pij_vectors = p_vector[:, rows] - p_vector[:, cols]
    vij_vectors = v_vector[:, rows] - v_vector[:, cols]

    pij_norm_pairs = pij_norm[rows, cols]
    vij_norm_pairs = vij_norm[rows, cols]
    pv_norm_pairs = pv_norm[rows, cols]
    
    # Pre-allocate result arrays (initialized to zero)
    h_x = np.zeros(matrix_size)
    Lfx = np.zeros(matrix_size)
    grad_vij = np.zeros((2, matrix_size))

    # --- 3. Build boolean masks for the if/else branches ---
    # sqrt_term is used by both VO and mask condition, so precompute
    # Note: NaN can occur when pij_norm_pairs < critical_distance (same assumption as before)
    

    
    sqrt_term_all = np.sqrt(pij_norm_pairs**2 - critical_distance**2)
    
    # 2. Compute VO expression
    # Gradient calculation
    grad_term = (sqrt_term_all / vij_norm_pairs)
    grad_vij = pij_vectors + grad_term[np.newaxis, :] * vij_vectors
        
    # h(x) calculation
    h_x = pv_norm_pairs + vij_norm_pairs * sqrt_term_all
        
    # Lfx calculation
    ca1 = vij_norm_pairs / sqrt_term_all
    Lfx = (vij_norm_pairs**2) + ca1 * pv_norm_pairs

    
    # --- 5. Apply margin ---
    h_x = h_x - margin

    # --- 6. Compute Lgh matrix (same as before, using completed grad_vij) ---
    g_i_all = V_CONST * np.vstack([-np.sin(seta), np.cos(seta)]) 
    g_j_all = V_CONST * np.vstack([np.sin(seta), -np.cos(seta)]) 
    
    g_i_pairs = g_i_all[:, rows]
    g_j_pairs = g_j_all[:, cols] 
    
    lg_i = np.sum(g_i_pairs * grad_vij, axis=0)
    lg_j = np.sum(g_j_pairs * grad_vij, axis=0)
    
    Lgh = np.zeros((matrix_size, N_AGENTS))
    row_indices_for_lgh = np.arange(matrix_size)
    Lgh[row_indices_for_lgh, rows] = lg_i
    Lgh[row_indices_for_lgh, cols] = lg_j

    # --- 7. Final return ---
    return h_x.reshape(-1, 1), Lfx.reshape(-1, 1), Lgh

In [ ]:
def solve_cbf_qp_with_warm_start(n, n_in, h_vector, lfh_vector, lgh_matrix,
                                 u_nominal, u_limit, class_k, t,
                                 qp_max_iter, qp_eps_abs, qp_time_limit):
    """
    Solve CBF-QP via DAQP directly.
    Returns (x, reason) where reason is one of:
        'ok', 'qp_nan', 'qp_infeasible', 'qp_iter_limit', 'qp_time_limit', 'qp_fail_<flag>'.
    """
    global _last_daqp_solve_time
    p_u = 0.5
    H = p_u * np.eye(n, dtype=np.float64)
    f = -p_u * u_nominal.flatten().astype(np.float64)

    A_cbf = (-lgh_matrix).astype(np.float64)
    h_cbf = (class_k * h_vector.flatten() + lfh_vector.flatten()).astype(np.float64)

    if np.any(np.isnan(A_cbf)) or np.any(np.isnan(h_cbf)) or np.any(np.isnan(f)):
        print(f"COLLISION (QP input NaN/Inf) at t={t}")
        _last_daqp_solve_time = 0.0; return None, 'collision'

    m = A_cbf.shape[0]
    bupper = np.concatenate([np.full(n, u_limit), h_cbf])
    blower = np.concatenate([np.full(n, -u_limit), np.full(m, -np.inf)])
    sense = np.zeros(n + m, dtype=np.int32)

    x, fval, flag, info = daqp.solve(
        H, f, A_cbf, bupper, blower, sense,
        primal_tol=float(qp_eps_abs), dual_tol=float(qp_eps_abs),
        iter_limit=int(qp_max_iter), time_limit=float(qp_time_limit),
    )

    _last_daqp_solve_time = float(info.get('solve_time', 0.0))
    if flag == 1 or flag == 2:
        return x.reshape(-1, 1), 'ok'
    reason = {-1: 'qp_infeasible', -2: 'qp_infeasible', -4: 'qp_iter_limit', -7: 'qp_time_limit'}.get(flag, f'qp_fail_{flag}')
    print(f"QP fail (daqp, flag={flag}, reason={reason}) at t={t}")
    return None, reason


In [ ]:
def calculate_std_dev(states_matrix, n_agents):
    """
    Computes the spatial standard deviation of a given state matrix (position or velocity).
    Implements the same logic as the MATLAB code: sqrt((1/N) * sum(diag(X_centered' * X_centered)))
    
    Args:
        states_matrix (np.ndarray): (2, N_AGENTS) position or velocity matrix.
        n_agents (int): number of agents.

    Returns:
        float: computed standard deviation.
    """
    if n_agents == 0:
        return 0.0
    
    # 1. Compute the center (mean) of the data.
    mean_vec = np.mean(states_matrix, axis=1, keepdims=True)
    
    # 2. Subtract the center from each point to center the data. (X_centered)
    centered_matrix = states_matrix - mean_vec
    
    # 3. Compute the sum of squared distances from each agent to the center.
    # np.sum(centered_matrix**2) equals sum(diag(X_centered.T @ X_centered)) and is more efficient.
    sum_of_squared_distances = np.sum(centered_matrix**2)
    
    # 4. Compute mean squared distance and take its square root to get the standard deviation (RMS distance).
    std_dev = np.sqrt(sum_of_squared_distances / n_agents)
    
    return std_dev

In [ ]:
def plot_initial_conditions(initial_states_matrix):
    """
    Visualizes the generated initial states.
    - Drones are represented as blue isosceles triangles following their heading.
    - The two closest drones are connected by a red dashed line.
    - The minimum distance is displayed at the top.

    Args:
        initial_states_matrix (np.ndarray): A 5xN_AGENTS matrix of initial states.
    """
    N_AGENTS = initial_states_matrix.shape[1]

    # Plot setup
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_title('Safe Initial States Visualization', fontsize=16)
    ax.set_xlabel('X Position (m)')
    ax.set_ylabel('Y Position (m)')
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, linestyle='--', alpha=0.6)

    min_dist = float('inf')
    closest_pair_indices = (-1, -1)

    # 1. Plot all drones as triangles
    for i in range(N_AGENTS):
        # Drone position and heading
        px, py = initial_states_matrix[0, i], initial_states_matrix[1, i]
        theta = initial_states_matrix[4, i]

        # Define triangle vertices (for heading 0)
        triangle_points = np.array([
            [10, 0],    # drone size
            [-10, 6],
            [-10, -6]
        ])

        # Rotate by heading angle (theta)
        rotation_matrix = np.array([
            [np.cos(theta), -np.sin(theta)],
            [np.sin(theta), np.cos(theta)]
        ])
        # (fix) Multiply by the transpose for correct rotation.
        rotated_points = triangle_points @ rotation_matrix.T

        # Translate to position (px, py)
        transformed_points = rotated_points + np.array([px, py])

        # Draw the triangle
        ax.add_patch(Polygon(transformed_points, closed=True, color='blue'))

    # 2. Find the two closest drones
    if N_AGENTS > 1:
        for i in range(N_AGENTS):
            for j in range(i + 1, N_AGENTS):
                pos_i = initial_states_matrix[0:2, i]
                pos_j = initial_states_matrix[0:2, j]
                distance = np.linalg.norm(pos_i - pos_j)
                
                if distance < min_dist:
                    min_dist = distance
                    closest_pair_indices = (i, j)

    # 3. Connect the closest drones and display the distance
    if closest_pair_indices[0] != -1:
        idx1, idx2 = closest_pair_indices
        p1 = initial_states_matrix[0:2, idx1]
        p2 = initial_states_matrix[0:2, idx2]

        # Connect with a red dashed line
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'r--', label=f'Min Distance: {min_dist:.2f} m')

        # Display minimum distance text at the top
        ax.text(0.5, 1.05, f'Minimum Distance: {min_dist:.2f} m', 
                ha='center', va='bottom', transform=ax.transAxes, 
                fontsize=14, color='red', fontweight='bold')

    # Set plot limits
    if N_AGENTS > 0:
        all_x = initial_states_matrix[0, :]
        all_y = initial_states_matrix[1, :]
        x_range = np.max(all_x) - np.min(all_x)
        y_range = np.max(all_y) - np.min(all_y)
        margin = max(x_range, y_range) * 0.1 if max(x_range, y_range) > 0 else 1
        ax.set_xlim(np.min(all_x) - margin, np.max(all_x) + margin)
        ax.set_ylim(np.min(all_y) - margin, np.max(all_y) + margin)
    else:
        ax.set_xlim(-10, 10)
        ax.set_ylim(-10, 10)

    # Show legend and plot
    if N_AGENTS > 1:
        ax.legend()
    plt.show()


In [ ]:

# ==============================================================================
# 2. Simulation loop (user code adapted for the 5D model)
# ==============================================================================
# Side-channel for per-call failure diagnostics.
run_status = {'reason': None, 'time': None}

def run_multi_agent_simulation(N_AGENTS,V_CONST, critical_distance,initial_states_matrix):
    # --- Simulation parameters ---
    
    # Feasibility flag
    feasible=1
    run_status['reason'] = 'ok'
    run_status['time'] = None

    # Compute std dev of initial position and velocity
    position_std_dev = calculate_std_dev(initial_states_matrix[0:2, :], N_AGENTS)
    velocity_std_dev = calculate_std_dev(initial_states_matrix[2:4, :], N_AGENTS)

    T_FINAL = 600
    DT_CONTROL = 0.05



    # -- Vehicle parameters --
    
    u_limit=0.35  # rad

    # -- ACS parameters 
    beta=np.load('parameter/beta.npy')
    lamda=np.load('parameter/lamda.npy')
    k1=np.load('parameter/k1.npy')
    k2=np.load('parameter/k2.npy')
    desired_distance=np.load(f'parameter/{CASE_ID}/desired_distance.npy')



    # CBF parameter
    
    class_k=np.load('parameter/class_k1.npy')
    margin = float(np.load('parameter/margin.npy'))
    qp_max_iter = int(np.load('parameter/qp_max_iter.npy'))
    qp_eps_abs = float(np.load('parameter/qp_eps_abs.npy'))
    qp_time_limit = float(np.load('parameter/qp_time_limit.npy'))
    fi_threshold = float(np.load('parameter/fi_threshold.npy'))   # forward-invariance failure threshold (centralized)

    # Problem dimension
    n=N_AGENTS
    n_eq = 0  # no equality constraints
    n_in=int(N_AGENTS*(N_AGENTS-1)/2)+N_AGENTS
    
    y0 = initial_states_matrix.flatten('F')

    # --- Simulation variables ---
    times = np.arange(0, T_FINAL, DT_CONTROL)
    history = [initial_states_matrix]
    control_history=[]
    minimum_distance_history=[]
    minimum_h_history=[]
    global _qp_time_history_buf, _qp_daqp_history_buf, _u_nominal_history_buf
    _qp_time_history_buf = []
    _qp_daqp_history_buf = []
    _u_nominal_history_buf = []
    maximum_distance_history=[]
    current_y = y0

    # --- NaN/Inf counter ---
    nan_inf_count = 0
    u_violation_count = 0

    print("5D Simulation starting...")
    
    for t in times[:-1]:

        # Print progress (every 50 s)
        if t > 0 and t % 50 < DT_CONTROL:
            print(f"  Progress: t={t:.0f}/{T_FINAL}s ({t/T_FINAL*100:.1f}%)")

        
        current_states_matrix = current_y.reshape((STATE_DIM, N_AGENTS), order='F')
        current_states_matrix[4, :] = (current_states_matrix[4, :] + np.pi) % (2 * np.pi) - np.pi
        
        # relative state information build
        pij_norm,vij_norm,pv_norm,seta_ij,p_dot_seta=relative_state(STATE_DIM, N_AGENTS,current_states_matrix)

        # Calculate minimum/maximum distance
        minimum_inter_distance=minimum_inter_distance_function(pij_norm,N_AGENTS)
        minimum_distance_history.append(minimum_inter_distance)

        # Collision detection (abort)
        if minimum_inter_distance < critical_distance:
            run_status['reason'] = 'collision'; run_status['time'] = float(t)
            print(f"COLLISION at t={t:.3f}: min_dist={minimum_inter_distance:.4f} < critical_distance={float(critical_distance):.4f}")
            break

        maximum_inter_distance=maximum_inter_distance_function(pij_norm,N_AGENTS)
        maximum_distance_history.append(maximum_inter_distance)

        # Control barrier function
        u_nominal=ACS_flocking(u_limit,beta,lamda,k1,k2,desired_distance ,N_AGENTS, V_CONST,pij_norm,vij_norm,pv_norm,seta_ij,p_dot_seta)

        h_vector,lfh_vector,lgh_matrix=vo_cbf(pij_norm, vij_norm, pv_norm,N_AGENTS, V_CONST,critical_distance,current_states_matrix,margin)

        # --- NaN/Inf check (count only, no termination) ---
        if np.any(np.isnan(h_vector)) or np.any(np.isinf(h_vector)) or \
           np.any(np.isnan(lfh_vector)) or np.any(np.isinf(lfh_vector)) or \
           np.any(np.isnan(lgh_matrix)) or np.any(np.isinf(lgh_matrix)):
            nan_inf_count += 1

        minimum_h = float(np.min(h_vector))
        minimum_h_history.append(minimum_h)

        # --- Forward invariance check: is h(x) below tolerance? ---
        if float(minimum_h) < fi_threshold:
            run_status['reason'] = 'forward_invariance_fail'; run_status['time'] = float(t)
            print(f"FORWARD_INVARIANCE_FAIL at t={t:.3f}: min h(x)={float(minimum_h):.3e} < {fi_threshold:.3e}")
            print(f"  [NaN/Inf occurrences: {nan_inf_count}]")
            feasible = 0
            return None, None, None, None, None, None, feasible, None, None, None, None

        # Solve QP
        _qp_t0 = time.perf_counter()
        optimal_u, qp_reason = solve_cbf_qp_with_warm_start(n, n_in,
         h_vector, lfh_vector, lgh_matrix, 
         u_nominal, u_limit, class_k, t, qp_max_iter, qp_eps_abs, qp_time_limit
        )
        _qp_time_history_buf.append(time.perf_counter() - _qp_t0)
        _qp_daqp_history_buf.append(_last_daqp_solve_time)
        _u_nominal_history_buf.append(u_nominal.copy())

        if optimal_u is None:
          run_status['reason'] = qp_reason; run_status['time'] = float(t)
          print(f"infeasibility occur: {qp_reason}")
          print(f"  [NaN/Inf occurrences: {nan_inf_count}]")
          feasible=0
          return None, None, None, None, None, None,feasible, None, None, None, None

        # --- u_limit violation check (diagnostic) ---
        max_abs_u = float(np.max(np.abs(optimal_u)))
        if max_abs_u > u_limit + qp_eps_abs:
            u_violation_count += 1
            if u_violation_count <= 5:
                print(f"U_LIMIT_VIOLATION at t={t:.3f}: |omega|_max={max_abs_u:.4f} > u_limit={u_limit:.4f}")

        control_history.append(optimal_u.copy())

        # Numerical integration
        sol = solve_ivp(
            fun=multi_agent_const_speed_5d_dynamics,
            t_span=[t, t + DT_CONTROL],
            y0=current_y,
            args=(optimal_u, N_AGENTS, STATE_DIM, V_CONST)
        )
        current_y = sol.y[:, -1]
        history.append(current_y.reshape((STATE_DIM, N_AGENTS), order='F'))
        
    
    print("Simulation finished.")
    print("feasible:", feasible)
    print(f"  [NaN/Inf occurrences: {nan_inf_count}]")
    print(f"  [u_limit violations: {u_violation_count}]")

    # Compute std dev of the final state
    final_states_matrix = history[-1]
    final_position_std_dev = calculate_std_dev(final_states_matrix[0:2, :], N_AGENTS)
    final_velocity_std_dev = calculate_std_dev(final_states_matrix[2:4, :], N_AGENTS)

    return (times, np.array(history), np.array(control_history),
            np.array(minimum_distance_history), np.array(maximum_distance_history),
            np.array(minimum_h_history), feasible, position_std_dev, velocity_std_dev,
            final_position_std_dev, final_velocity_std_dev)


In [ ]:
# --- Initial state setup (5-dimensional) ---

N_AGENTS =np.load(f'parameter/{CASE_ID}/N_AGENTS.npy')
V_CONST = np.load('parameter/V_CONST.npy')
critical_distance=np.load(f'parameter/{CASE_ID}/critical_distance.npy')
STATE_DIM = np.load('parameter/STATE_DIM.npy')
initial_test_case=np.load(f'initial_conditions/{CASE_ID}/initial.npy')
test_case_num = int(initial_test_case.shape[-1])   # derived from initial.npy shape

In [ ]:
vo_total_history=[]
vo_total_control_history=[]
vo_total_minimum_distance_history=[]
vo_total_maximum_distance_history=[]
vo_total_minimum_h_history=[]
vo_total_qp_time_list=[]
vo_total_daqp_solve_time_list=[]
vo_total_u_nominal_history=[]
vo_feasibility_list=[]
vo_initial_position_std_dev_list=[]
vo_initial_velocity_std_dev_list=[]
vo_final_position_std_dev_list=[]
vo_final_velocity_std_dev_list=[]

In [ ]:
# Run simulation
vo_failure_reason_list = []
vo_failure_time_list = []
for test_case in range(test_case_num):
    print(f"Running simulation for test case {test_case }...")
    initial_states_matrix=initial_test_case[:,:,test_case]
    times, history,heading_angel_rate,smallist_distance ,largiest_distance,smallist_h,feasiblity,initial_position_std_dev, initial_velocity_std_dev, final_position_std_dev, final_velocity_std_dev = run_multi_agent_simulation(N_AGENTS,V_CONST, critical_distance,initial_states_matrix)
    vo_failure_reason_list.append(run_status['reason'])
    vo_failure_time_list.append(run_status['time'])
    
    if feasiblity==1:
        print(f"Test case {test_case} is feasible and recorded.")
        vo_total_history.append(history)
        vo_total_control_history.append(heading_angel_rate)
        vo_total_minimum_distance_history.append(smallist_distance)
        vo_total_maximum_distance_history.append(largiest_distance)
        vo_total_minimum_h_history.append(smallist_h)
        vo_total_qp_time_list.append(list(_qp_time_history_buf))
        vo_total_daqp_solve_time_list.append(list(_qp_daqp_history_buf))
        vo_total_u_nominal_history.append(np.array(_u_nominal_history_buf))
        vo_feasibility_list.append(True)
        vo_initial_position_std_dev_list.append(initial_position_std_dev)
        vo_initial_velocity_std_dev_list.append(initial_velocity_std_dev)
        vo_final_position_std_dev_list.append(final_position_std_dev)
        vo_final_velocity_std_dev_list.append(final_velocity_std_dev)
    else:
        print(f"Test case {test_case} is infeasible.")
        vo_total_history.append(None)
        vo_total_control_history.append(None)
        vo_total_minimum_distance_history.append(None)
        vo_total_maximum_distance_history.append(None)
        vo_total_minimum_h_history.append(None)
        vo_total_qp_time_list.append(None)
        vo_total_daqp_solve_time_list.append(None)
        vo_total_u_nominal_history.append(None)
        vo_feasibility_list.append(False)
        vo_initial_position_std_dev_list.append(None)
        vo_initial_velocity_std_dev_list.append(None)
        vo_final_position_std_dev_list.append(None)
        vo_final_velocity_std_dev_list.append(None)

    feasible_count = sum(vo_feasibility_list)
    print(f"\n--- Simulation Summary ---")
    print(f"Feasible cases: {feasible_count} / {test_case_num}")
    if test_case_num > 0:
        feasibility_rate = (feasible_count / test_case_num) * 100
        print(f"Feasibility Rate: {feasibility_rate:.2f}%")

In [ ]:
# 


vo_data={

    'vo_total_history':vo_total_history,
    'vo_total_control_history':vo_total_control_history,
    'vo_total_minimum_distance_history':vo_total_minimum_distance_history,
    'vo_total_maximum_distance_history':vo_total_maximum_distance_history,
    'vo_total_minimum_h_history':vo_total_minimum_h_history,
    'vo_total_qp_time_list':vo_total_qp_time_list,
    'vo_total_daqp_solve_time_list':vo_total_daqp_solve_time_list,
    'vo_total_u_nominal_history':vo_total_u_nominal_history,
    'vo_feasibility_list':vo_feasibility_list,
    'vo_initial_position_std_dev_list':vo_initial_position_std_dev_list,
    'vo_initial_velocity_std_dev_list':vo_initial_velocity_std_dev_list,
    'vo_final_position_std_dev_list':vo_final_position_std_dev_list,
    'vo_final_velocity_std_dev_list':vo_final_velocity_std_dev_list,

    'vo_failure_reason_list': vo_failure_reason_list,
    'vo_failure_time_list': vo_failure_time_list,
}

with open(f"result/{CASE_ID}/vo/vo_simulation_data.pkl", "wb") as f:
    pickle.dump(vo_data, f)

    print("Saved! You should see vo_simulation_data.pkl in the folder.")